# Vision Transformer (ViT)

**Paper:**  
Dosovitskiy, Alexey et al.  
*An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*  
ICLR 2021.

**arXiv:**  
https://arxiv.org/abs/2010.11929

This paragraph describes the core idea behind **Vision Transformers (ViT)**: how to convert an image into something a Transformer can process.

Original Transformers were designed for text, where the input is a 1D sequence of word/token embeddings. Images, however, are naturally 2D structures (height × width).  

ViT solves this by:

- splitting the image into small patches,
- converting each patch into a vector,
- and treating those patches like "tokens" or "words".

<div>
    <img src='../../../images/ViT.png' width="800">
</div>

---

# 1. Original Image Representation

The image is represented as:

$$
x \in \mathbb{R}^{H \times W \times C}
$$

Where:

- $H$ = image height
- $W$ = image width
- $C$ = number of channels

For example:

- RGB image → $C = 3$

A 224×224 RGB image would have shape:

$$
224 \times 224 \times 3
$$

---

# 2. Splitting the Image into Patches

The image is divided into small square patches of size:

$$
(P, P)
$$

For example:

- patch size = 16×16

Then:

- a 224×224 image
- with 16×16 patches

produces:

$$
\frac{224 \times 224}{16 \times 16} = 196
$$

patches.

The general formula is:

$$
N = \frac{HW}{P^2}
$$

Where:

- $N$ = total number of patches.

---

# 3. Flattening Each Patch

Each patch has size:

$$
P \times P \times C
$$

If:

- $P = 16$
- $C = 3$

then each patch contains:

$$
16 \times 16 \times 3 = 768
$$

values.

Each patch is then flattened into a 1D vector:

$$
x_p \in \mathbb{R}^{N \times (P^2 C)}
$$

So:

- there are $N$ patches,
- and each patch becomes a vector.

This is equivalent to transforming the image into a sequence of tokens.

---

# 4. Linear Projection

The Transformer requires all tokens to have the same embedding dimension \(D\).

Therefore, each flattened patch is passed through a trainable linear layer:

$$
\text{patch} \rightarrow D
$$

For example:

- flattened patch: 768 dimensions
- projected into:
  - 512
  - 768
  - 1024
  - etc.

depending on the model configuration.

This operation is simply a learnable matrix multiplication.

---

# 5. Patch Embeddings

The final vectors are called:

> patch embeddings

because they play the same role as word embeddings in NLP.

The Transformer now receives:

$$
[\text{patch}_1,\text{patch}_2,\ldots,\text{patch}_N]
$$

as an input sequence.

---

# Important Intuition

## CNNs

- use local kernels,
- apply convolutions,
- have strong spatial inductive biases.

## Vision Transformers (ViT)

- do not rely on deep convolutions,
- only split images into patches,
- and let self-attention learn global relationships.

This is why CNNs have stronger inductive bias:
they already assume spatial locality and translation invariance.

ViTs learn these patterns more directly from data, but usually require much larger datasets.

# 6. The `[class]` Token

ViT introduces a special learnable token, similar to the `[CLS]` token used in BERT.

Before feeding the patch embeddings into the Transformer, an extra embedding is added at the beginning of the sequence:

$$
z_0^0 = x_{\text{class}}
$$

So instead of:

$$
[\text{patch}_1, \text{patch}_2, \ldots, \text{patch}_N]
$$

the input becomes:

$$
[x_{\text{class}}, \text{patch}_1, \text{patch}_2, \ldots, \text{patch}_N]
$$

The key idea is:

- this token does not correspond to any image patch,
- it is a learnable vector,
- and through self-attention it gathers information from all patches.

---

# 7. Why Use the Class Token?

During Transformer processing:

- every token attends to every other token,
- including the class token.

As information flows through the layers, the class token gradually becomes a global summary of the entire image.

After the final Transformer layer \(L\), the output representation of the class token is:

$$
z_L^0
$$

This vector is used as the final image representation:

$$
y = z_L^0
$$

In practice:

- the entire image information gets compressed into this vector,
- and classification is performed using this representation.

This is very similar to BERT, where the `[CLS]` token summarizes an entire sentence.

---

# 8. Classification Head

After obtaining the final class token representation, a classification head is attached.

## During Pretraining

They use:

- an MLP (Multi-Layer Perceptron),
- with one hidden layer.

This gives the model more flexibility during large-scale training.

Structure:

$$
z_L^0 \rightarrow \text{MLP} \rightarrow \text{class logits}
$$

---

## During Fine-Tuning

They simplify the head to:

- a single linear layer.

Structure:

$$
z_L^0 \rightarrow \text{Linear Layer} \rightarrow \text{class logits}
$$

Why?

Because the Transformer already learned powerful representations during pretraining, so a simpler classifier is sufficient during transfer learning.

---

# 9. Positional Embeddings

Transformers themselves do not understand spatial order.

If we only provide patches:

$$
[\text{patch}_1, \text{patch}_2, \ldots]
$$

the model would not know:

- which patch comes first,
- which patch is above another,
- or where objects are located.

This is because self-attention is permutation-invariant.

---

# 10. Adding Positional Information

To preserve spatial structure, ViT adds positional embeddings to each patch embedding.

Conceptually:

$$
\text{input embedding} =
\text{patch embedding} +
\text{position embedding}
$$

Each position has its own learnable vector.

For example:

- patch 1 → receives position embedding 1
- patch 2 → receives position embedding 2
- etc.

This allows the Transformer to learn spatial relationships.

---

# 11. Why 1D Positional Embeddings?

Even though images are 2D, the paper uses standard 1D learnable positional embeddings.

Why?

Because after flattening patches into a sequence, the image is effectively treated as:

$$
[\text{patch}_1, \text{patch}_2, \ldots, \text{patch}_N]
$$

a 1D token sequence.

The authors mention that more sophisticated 2D-aware positional embeddings did not significantly improve performance.

So they kept the simpler approach.

---

# Full ViT Pipeline So Far

The complete flow becomes:

1. Input image
2. Split into patches
3. Flatten patches
4. Linear projection → patch embeddings
5. Add class token
6. Add positional embeddings
7. Feed sequence into Transformer encoder
8. Extract final class token representation
9. Classification head predicts the image class

Conceptually:

$$
\text{Image}
\rightarrow
\text{Patches}
\rightarrow
\text{Embeddings}
\rightarrow
\text{Transformer}
\rightarrow
\text{Class Token}
\rightarrow
\text{Classifier}
$$

# 12. Transformer Encoder in ViT

After constructing the input sequence with:

- patch embeddings,
- the class token,
- and positional embeddings,

the sequence is passed through a standard Transformer encoder.

The encoder is composed of repeated blocks containing:

- Multi-Head Self-Attention (MSA),
- MLP layers,
- LayerNorm,
- and residual connections.

---

# 13. Initial Input Sequence

The input to the Transformer is:

$$
z_0 =
[x_{\text{class}}; x_p^1E; x_p^2E; \ldots; x_p^NE]
+ E_{\text{pos}}
$$

Where:

- $x_{\text{class}}$ = learnable class token
- $x_p^i$ = flattened image patch
- $E$ = learnable linear projection matrix
- $E_{\text{pos}}$ = positional embeddings

This produces the complete sequence of embeddings processed by the Transformer.

---

# 14. Multi-Head Self-Attention Block

The first operation inside each Transformer layer is:

$$
z'_\ell =
\text{MSA}(\text{LN}(z_{\ell-1}))
+
z_{\ell-1}
$$

Where:

- LN = Layer Normalization
- MSA = Multi-Head Self-Attention
- $z_{\ell-1}$ = previous layer output

---

## What Happens Here

### LayerNorm

The input is normalized before attention.

This improves:
- stability,
- optimization,
- and gradient flow.

---

### Multi-Head Self-Attention

Each token attends to every other token.

This means:

- patches exchange information globally,
- long-range spatial relationships are learned,
- and the class token gathers information from the entire image.

Unlike CNNs, which process images locally, self-attention allows direct global interactions.

---

### Residual Connection

The original input is added back:

$$
+ z_{\ell-1}
$$

This skip connection helps train deep networks effectively.

---

# 15. MLP Block

After attention, the output passes through an MLP block:

$$
z_\ell =
\text{MLP}(\text{LN}(z'_\ell))
+
z'_\ell
$$

The MLP consists of:

```text
Linear
→ GELU
→ Linear
```

Where GELU is the activation function commonly used in Transformers.

---

# 16. Repeated Transformer Layers

The Transformer encoder repeats this process \(L\) times:

```text
LayerNorm
→ Multi-Head Self-Attention
→ Residual Add
→ LayerNorm
→ MLP
→ Residual Add
```

Each layer progressively builds more abstract and global image representations.

---

# 17. Final Image Representation

After the final Transformer layer, the output corresponding to the class token is extracted:

$$
y = \text{LN}(z_L^0)
$$

This vector becomes the final image representation.

It is then passed to the classification head to predict image classes.

---

# Overall Intuition

ViT works by:

1. Converting the image into a sequence of patch tokens
2. Processing the sequence using a standard Transformer encoder
3. Using the final class token as a global image representation
4. Performing classification from that representation

The key innovation is that ViT applies an almost pure NLP-style Transformer architecture directly to images.